In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import requests
import time
from io import StringIO

# I. Load DPCstruct Mcs Sequences

In [2]:
# DPCStruct Mcs Sequences
df1 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_mcs_sequences.csv")
df1.head()

,mc_id,protein_id,prot_range,prot_seq
0,MC0,A0A1Q3ZAL7,4-118,LIFLFVLTTTSFTTNLFKAQVNVNININSQPEWGPRGYNYVESYYL...
1,MC0,A0A537IMV5,3-112,KIFICFALCLLAAFFKPASAQFSTNENIKDQPKWGLAGQKYVEYYY...
2,MC0,A0A170YWQ7,2-107,NaN
3,MC0,A0A3E2NVG7,1-133,NaN
4,MC0,A0A2W4V3S2,3-113,NaN


In [3]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1642061 entries, 0 to 1642060
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   mc_id       1642061 non-null  object
 1   protein_id  1642061 non-null  object
 2   prot_range  1642061 non-null  object
 3   prot_seq    56438 non-null    object
dtypes: object(4)
memory usage: 50.1+ MB


# I. Check which protein IDs already exist in DPCFam dataframe source

In [4]:
# We use the DPCfam dataframe with info about proteins in UniProtKB 
df2 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/uniref50_proteins.csv")
df2.head()

,uniref50_id,uniprotkb_id,uniprotkb_accession,length
0,Q8WZ42-5,Q8WZ42-5,Q8WZ42-5,32900
1,Q3ASY8,Q3ASY8_CHLCH,Q3ASY8,36805
2,G5B0U1,G5B0U1_HETGA,G5B0U1,36507
3,Q8WZ42,TITIN_HUMAN,Q9Y6L9,34350
4,K7EE71,K7EE71_ORNAN,K7EE71,7472


In [5]:
# Info
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23531980 entries, 0 to 23531979
Data columns (total 4 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   uniref50_id          object
 1   uniprotkb_id         object
 2   uniprotkb_accession  object
 3   length               int64 
dtypes: int64(1), object(3)
memory usage: 718.1+ MB


In [6]:
# How many unique protein IDs are there in uniref50_id column?
print("Number of unique protein IDs in uniref50_id column:", df2["uniref50_id"].nunique())
# How many unique protein IDs are there in uniprotkb_accession column?
print("Number of unique protein IDs in uniprotkb_accession column:", df2["uniprotkb_accession"].nunique())
# How many unique protein IDs are there in their intersection?
intersection = set(df2["uniref50_id"]).intersection(set(df2["uniprotkb_accession"]))
print("Number of unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns:", len(intersection))
# Print the first 5 unique protein IDs in the intersection
print("First 5 unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns:", list(intersection)[:5])
# How many samples are there in their difference?
difference = set(df2["uniref50_id"]).difference(set(df2["uniprotkb_accession"]))
print("Number of unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns:", len(difference))
# Print the first 5 unique protein IDs in the difference
print("First 5 unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns:", list(difference)[:5])

Number of unique protein IDs in uniref50_id column: 23531980
Number of unique protein IDs in uniprotkb_accession column: 19681201
Number of unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns: 19588018
First 5 unique protein IDs in the intersection of uniref50_id and uniprotkb_accession columns: ['A0A0M8WLX9', 'D9PND5', 'N0DV06', 'A0A1J1HZ71', 'D3EHS2']
Number of unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns: 3943962
First 5 unique protein IDs in the difference between uniref50_id and uniprotkb_accession columns: ['UPI00069C8AD5', 'UPI0003627244', 'UPI0009DF4706', 'UPI0008F9B8E2', 'UPI000A0624CA']


In [7]:
# Let's  create a subdataframe with only that intersection of protein IDs
df_proteins_intersection = df2[df2["uniref50_id"].isin(intersection) & df2["uniprotkb_accession"].isin(intersection)]
df_proteins_intersection.head()

,uniref50_id,uniprotkb_id,uniprotkb_accession,length
0,Q8WZ42-5,Q8WZ42-5,Q8WZ42-5,32900
1,Q3ASY8,Q3ASY8_CHLCH,Q3ASY8,36805
2,G5B0U1,G5B0U1_HETGA,G5B0U1,36507
4,K7EE71,K7EE71_ORNAN,K7EE71,7472
5,W5MH34,W5MH34_LEPOC,W5MH34,32359


In [8]:
df_proteins_intersection.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19588018 entries, 0 to 23531979
Data columns (total 4 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   uniref50_id          object
 1   uniprotkb_id         object
 2   uniprotkb_accession  object
 3   length               int64 
dtypes: int64(1), object(3)
memory usage: 747.2+ MB


In [9]:
# We save that dataframe with the intersection of protein IDs in a new CSV file
df_proteins_intersection.to_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/uniref50_proteins_intersection.csv", index=False)

**1. We compare protein IDs in DPCStruct with the intersection of UniRef50 and UniProtKB**

In [10]:
# Now, let's consider protein IDs in df1 and check how many of them are in the intersection of protein IDs in df2
protein_ids_df1 = set(df1["protein_id"])
print("Number of unique protein IDs in df1:", len(protein_ids_df1))
intersection_df1 = protein_ids_df1.intersection(intersection)
print("Number of unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2:", len(intersection_df1))
# Print the first 5 unique protein IDs in the intersection of df1 and the intersection of uniref50_id and uniprotkb_accession columns in df2    
print("First 5 unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2:", list(intersection_df1)[:5])

Number of unique protein IDs in df1: 1252179
Number of unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2: 315482
First 5 unique protein IDs in df1 that are in the intersection of uniref50_id and uniprotkb_accession columns in df2: ['M1YVL6', 'A0A0F8Y0Y6', 'E4WU83', 'A0A1Q6UX40', 'A0A0Q8PLC3']


**2. We compare protein IDs in DPCStruct with Uniref50**

In [11]:
# How many protein IDs in df1 are in uniref50_id column in df2?
intersection_df1_uniref50 = protein_ids_df1.intersection(set(df2["uniref50_id"]))
print("Number of unique protein IDs in df1 that are in uniref50_id column in df2:", len(intersection_df1_uniref50))
# Print the first 5 unique protein IDs in the intersection of df1 and uniref50_id column in df2    
print("First 5 unique protein IDs in df1 that are in uniref50_id column in df2:", list(intersection_df1_uniref50)[:5])

Number of unique protein IDs in df1 that are in uniref50_id column in df2: 316461
First 5 unique protein IDs in df1 that are in uniref50_id column in df2: ['M1YVL6', 'A0A0F8Y0Y6', 'E4WU83', 'A0A1Q6UX40', 'A0A0Q8PLC3']


**3.We compare protein IDs in DPCstruct with UniProtKB Accession Number**

In [12]:
# How many protein IDs in df1 are in uniprotkb_accession column in df2?
intersection_df1_uniprotkb = protein_ids_df1.intersection(set(df2["uniprotkb_accession"]))
print("Number of unique protein IDs in df1 that are in uniprotkb_accession column in df2:", len(intersection_df1_uniprotkb))
# Print the first 5 unique protein IDs in the intersection of df1 and uniprotkb_accession column in df2    
print("First 5 unique protein IDs in df1 that are in uniprotkb_accession column in df2:", list(intersection_df1_uniprotkb)[:5])

Number of unique protein IDs in df1 that are in uniprotkb_accession column in df2: 315482
First 5 unique protein IDs in df1 that are in uniprotkb_accession column in df2: ['M1YVL6', 'A0A0F8Y0Y6', 'E4WU83', 'A0A1Q6UX40', 'A0A0Q8PLC3']


**Conclusion**:

- We have `1,642,061` domains distributed over `1,252,179` proteins, organised into `28,246` metaclusters.

- `315,482` unique protein IDs are available in both `UniRef50` and `UniProtKB Accession Number`: meaning we have the lengths of `25.19%` of the total of new proteins.

- `316,461` unique protein IDs are available in `UniRef50`: meaning we have the lengths of `25.27%` of the total of new proteins.

- `315,482` unique protein IDs are available in `UniProtKB Accession Number`, matching the `intersection`.

- We are focused on obtaining all lengths. Therefore, we create a subdataframe `[protein_id,length]` where the `length` is already known in `UniRef50`.

# III. Mapping each unique protein ID to its length

In [13]:
# We build a dataframe df3 with columns "protein" and "length", coming from the intersection of protein IDs in df1 and uniref50_id column in df2
df3 = df2[df2["uniref50_id"].isin(intersection_df1_uniref50)][["uniref50_id", "length"]].rename(columns={"uniref50_id": "protein_id", "length": "length"})
# We reset the index of df3
df3 = df3.reset_index(drop=True)
df3.head()

,protein_id,length
0,A0A090M7B6,892
1,F1MAE5,895
2,B8JI51,1992
3,K1PN73,98
4,M1CI46,681


In [14]:
# Info
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316461 entries, 0 to 316460
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   protein_id  316461 non-null  object
 1   length      316461 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 4.8+ MB


In [15]:
# Great, for now, we know 25% aabout the lengths of dpcstruct peoteins
# Let's save that dataframe with the lengths of dpcstruct proteins in a new CSV file
df3.to_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_protein_df1_lengths.csv", index=False)

In [16]:
# What are the remaining proteins IDs for which we do not have the lengths in df1?
remaining_protein_ids = protein_ids_df1.difference(set(df3["protein_id"]))
print("Number of remaining protein IDs in df1 for which we do not have the lengths in df3:", len(remaining_protein_ids))
# Print the first 5 remaining protein IDs
print("First 5 remaining protein IDs in df1 for which we do not have the lengths in df3:", list(remaining_protein_ids)[:5])

Number of remaining protein IDs in df1 for which we do not have the lengths in df3: 935718
First 5 remaining protein IDs in df1 for which we do not have the lengths in df3: ['A0A6L9LIM2', 'A0A377HSE9', 'A0A822STC8', 'A0A420IZ74', 'A0A377BDC4']


In [17]:
# We crate a new dataframe df4 with the remaining protein IDs. Columns are "protein_id" and "length", where length is set to NaN for now
df4 = pd.DataFrame({"protein_id": list(remaining_protein_ids), "length": np.nan})
df4.head()  

,protein_id,length
0,A0A6L9LIM2,NaN
1,A0A377HSE9,NaN
2,A0A822STC8,NaN
3,A0A420IZ74,NaN
4,A0A377BDC4,NaN


In [18]:
# Info
df4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 935718 entries, 0 to 935717
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   protein_id  935718 non-null  object 
 1   length      0 non-null       float64
dtypes: float64(1), object(1)
memory usage: 14.3+ MB


In [19]:
# Great, we are missing  the lengths of 75% of dpcstruct proteins. 
# We will fetch those lengths from UniProt in the next steps. 
# For now, we save that dataframe with the remaining protein IDs and NaN lengths in a new CSV file
df4.to_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_protein_df2_remaining.csv", index=False)

# IV. We retrieve all missing lengths from UniProt

`Step 1` : The running of our script `fetch_uniprot_complete.py` was failing after the 7th iteration. So, step 1, is considered as failed.

`Step 2` : We download the latest release of sequences from UniProt : 
    
-  mkdir -p /u/mdmc/enyanduk/internship_areasciencepark/Data/uniprot_fasta

-  `cd /u/mdmc/enyanduk/internship_areasciencepark/Data/uniprot_fasta`

- Swiss-Prot (~90 MB — takes seconds)
`wget -c https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz`

- TrEMBL (~55 GB — it takes a while)
`wget -c https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_trembl.fasta.gz`

- `python3 /u/mdmc/enyanduk/internship_areasciencepark/dpc_fam_and_struct_webapp/static/scripts/dpcstruct/extract_lengths_from_fasta.py`

`Step 3` : Step 2 missed around `30%` lengths. So, we communicate with UniProt API : `python3 fetch_missing_lengths.py`, its successfull completion produced all missing lengths.

In [20]:
df5 = pd.read_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_protein_lengths_complete.csv")
df5.head()

,protein_id,length
0,A0A009ECR5,431
1,A0A009EYE4,338
2,A0A009F4T8,254
3,A0A009G5I8,218
4,A0A009H1M6,174


In [21]:
# Info
df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 935718 entries, 0 to 935717
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   protein_id  935718 non-null  object
 1   length      935718 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 14.3+ MB


In [22]:
# We know that df3 (known from uniref50) and df5 (known from uniprot) are disjoint sets, a quick check confirms that
intersection_df3_df5 = set(df3["protein_id"]).intersection(set(df5["protein_id"]))
print("Number of unique protein IDs in the intersection of df3 and df5:", len(intersection_df3_df5))

Number of unique protein IDs in the intersection of df3 and df5: 0


In [23]:
# Great, let's merge df3 and df5 to get a complete dataframe with the lengths of all dpcstruct proteins, and save that dataframe in a new CSV file
df = pd.concat([df3, df5], ignore_index=True)
df.head()

,protein_id,length
0,A0A090M7B6,892
1,F1MAE5,895
2,B8JI51,1992
3,K1PN73,98
4,M1CI46,681


In [24]:
# Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1252179 entries, 0 to 1252178
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   protein_id  1252179 non-null  object
 1   length      1252179 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 19.1+ MB


In [25]:
# We save that complete dataframe with the lengths of all dpcstruct proteins in a new CSV file
df.to_csv("/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_all_proteins_lengths.csv", index=False)